In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

In [ ]:
import os
import torch
from pprint import pprint
from typing import Optional
from recipes.research.ar import ARUMM
from recipes.research.diff import DiffUMM
from recipes.research.audio_codec.scripts.compile import get_latest_model_from_commit

def get_ar_umm(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("ar/default", commit_hash)
        
    model = ARUMM.load_from_checkpoint(ckpt_path)
    model = model.eval().to("cuda")
    print(model.summarize())
    pprint(model.config)
    model.commit_hash = commit_hash
    model.commit_step = os.path.basename(ckpt_path)
    return model


def get_diff_umm(commit_hash: str, ckpt_path: Optional[str] = None):
    if ckpt_path is None:
        ckpt_path = get_latest_model_from_commit("diff/default", commit_hash)
        
    diff = DiffUMM.load_from_checkpoint(ckpt_path)
    diff = diff.eval().to("cuda")
    print(diff.summarize())
    pprint(diff.config)
    diff.commit_hash = commit_hash
    diff.commit_step = os.path.basename(ckpt_path)
    return diff


In [ ]:
model = get_ar_umm("a4a9034")
model = model.to("cuda")
model.setup()

# UMM Diffusion

In [ ]:
diff = get_diff_umm("ee7af12")
diff = diff.to("cuda")
diff.setup()

In [ ]:
from samantha.data.utils import read_audio

fp = "/mnt/bn/janne-research-xl/data/instrumental_hq/Cinéma/1978) Superman (John Williams)/cd 1/01 - Prelude And Main Title March.mp3"
audio, sr = read_audio(fp, diff.config.sample_rate, normalize_loudness=True)
audio = audio.to("cuda")
with torch.no_grad():
    pred_noise = diff.sample_from_audio(audio, sr, t=50, cfg_weight=1.1, schedule_tau=0.4)
    pred_audio = diff.decode_audio(pred_noise.permute(0, 2, 1))

In [ ]:
from IPython.display import Audio, display
display(Audio(pred_audio[0].cpu(), rate=diff.config.sample_rate))

# UMMAR

In [ ]:
from samantha.data.audio.dataset import AudioFolderDataset, AudioFolderDataModule

sample_rate = diff.config.sample_rate
dataset = AudioFolderDataset(
    root="/mnt/bn/janne-research-xl/data/instrumental_hq/",
    pad=True,
    duration=10,
    sample_rate=sample_rate,
    shuffle=True,
    num_channels=2,
    num_workers=16
)
datamodule = AudioFolderDataModule([dataset], [dataset], [dataset], weights=None, batch_size=8, shuffle=True, num_workers=16)
train_loader = datamodule.train_dataloader()

In [ ]:
batch = next(iter(train_loader))

In [ ]:
from IPython.display import display, Audio

batch_idx = 2
audio = batch.audio[batch_idx:batch_idx+1].to("cuda")
display(Audio(audio[0].cpu(), rate=diff.config.sample_rate))

In [ ]:
with torch.no_grad():
    ar_result = model.sample(audio, diff.config.sample_rate, temperature=0.1, do_sample=True)
    pred_ar_noise = diff.sample_from_umm_tokens(ar_result.sampled_token_ids, t=50, cfg_weight=1.1, schedule_tau=0.4)
    pred_ar_audio = diff.decode_audio(pred_ar_noise.permute(0, 2, 1))
    gt_audio = diff.decode_audio(diff.get_audio_latents(audio, diff.config.sample_rate))

In [ ]:
from IPython.display import Audio, display
display(Audio(gt_audio[0].cpu(), rate=diff.config.sample_rate))
display(Audio(pred_ar_audio[0].cpu(), rate=diff.config.sample_rate))

In [ ]:
print(ar_result.sampled_token_ids[:, 50:])
print(ar_result.token_ids[:, 50:])

In [ ]:
from samantha.transforms.audio import batch_plot_spectrogram

gt_mel = model.umm.token2mel(ar_result.token_ids)
pred_mel = model.umm.token2mel(ar_result.sampled_token_ids)

batch_plot_spectrogram(gt_mel.permute(0, 2, 1), plot_log=False)
batch_plot_spectrogram(pred_mel.permute(0, 2, 1), plot_log=False)